# DeepSplat live webcam segmentation
Select a GPU runtime and enable notebook access to your HF_TOKEN Colab secret. Run all cells, then click Start camera in the output.


In [ ]:
!nvidia-smi
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install -q 'git+https://github.com/facebookresearch/sam3.git@eef6839bcfc3458dd52ea80535dc6bd62aea86b1' 'numpy==1.26.4' 'opencv-python-headless==4.11.0.86' pycocotools einops psutil


In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

checkpoint = hf_hub_download("facebook/sam3.1", "sam3.1_multiplex.pt", token=userdata.get("HF_TOKEN"))


In [ ]:
from unittest.mock import patch
import torch
from sam3.model_builder import build_sam3_multiplex_video_predictor

assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name())
autocast = torch.autocast

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Autocast dtype: {dtype}')

def runtime_autocast(*args, **kwargs):
    if kwargs.get("dtype") == torch.bfloat16:
        kwargs["dtype"] = dtype
    return autocast(*args, **kwargs)

with patch.object(torch, "autocast", runtime_autocast):
    predictor = build_sam3_multiplex_video_predictor(
        checkpoint_path=checkpoint, use_fa3=False, use_rope_real=False
    )
predictor.bf16_context.__exit__(None, None, None)
model = predictor.model


In [ ]:
from base64 import b64decode, b64encode
from io import BytesIO

import cv2
import numpy as np
from PIL import Image
from google.colab import output
from IPython.display import JSON, Javascript, display

def segment_frame(data_url, prompt_text):
    prompts = [line.strip() for line in prompt_text.splitlines() if line.strip()]
    if not 1 <= len(prompts) <= 8:
        raise ValueError("Enter one to eight prompts, one per line.")
    with Image.open(BytesIO(b64decode(data_url.split(",", 1)[1]))) as source:
        image = source.convert("RGB")
    overlay = np.array(image)
    colors = [(255, 75, 85), (70, 220, 100), (245, 165, 50)]
    with torch.inference_mode(), autocast("cuda", dtype=dtype):
        for index, prompt in enumerate(prompts):
            state = model.init_state(resource_path=[image])
            _, result = model.add_prompt(
                state, frame_idx=0, text_str=prompt, output_prob_thresh=.5
            )
            del state
            color = colors[index % len(colors)]
            for mask, score in zip(result["out_binary_masks"], result["out_probs"]):
                region = np.asarray(mask, dtype=bool).reshape(overlay.shape[:2])
                rows, cols = np.nonzero(region)
                if not rows.size:
                    continue
                overlay[region] = .55 * overlay[region] + .45 * np.array(color)
                cv2.putText(overlay, f"{prompt} {score:.2f}",
                            (int(cols.min()), max(22, int(rows.min()))),
                            cv2.FONT_HERSHEY_SIMPLEX, .65, color, 2, cv2.LINE_AA)
    buffer = BytesIO()
    Image.fromarray(overlay).save(buffer, format="JPEG", quality=85)
    return JSON({"image": "data:image/jpeg;base64," + b64encode(buffer.getvalue()).decode("ascii")})

output.register_callback("deepsplat.segment_frame", segment_frame)
display(Javascript(r"""
(() => {
  const root = document.createElement('section');
  root.innerHTML = `<label>Defect prompts (one per line)<br><textarea rows="3">peeling paint</textarea></label>
    <p><button>Start camera</button> <button disabled>Stop camera</button> <span role="status"></span></p>
    <div style="display:flex;gap:12px;flex-wrap:wrap">
      <video autoplay muted playsinline style="width:min(420px,100%)"></video>
      <img alt="Latest segmentation frame" style="width:min(420px,100%);object-fit:contain">
    </div>`;
  document.body.append(root);
  const [start, stop] = root.querySelectorAll('button');
  const video = root.querySelector('video');
  const image = root.querySelector('img');
  const status = root.querySelector('span');
  const prompts = root.querySelector('textarea');
  const canvas = document.createElement('canvas');
  const context = canvas.getContext('2d');
  const nextFrame = () => new Promise(resolve => requestAnimationFrame(resolve));
  let stream, running = false;
  function stopCamera() {
    running = false;
    stop.disabled = true;
    stream?.getTracks().forEach(track => track.stop());
    video.srcObject = null;
  }
  stop.onclick = stopCamera;
  window.addEventListener('pagehide', stopCamera, {once: true});
  start.onclick = async () => {
    start.disabled = true;
    try {
      stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
      video.srcObject = stream;
      await video.play();
      running = true;
      stop.disabled = false;
      while (running) {
        await nextFrame();
        if (!running || !video.videoWidth) continue;
        if (!prompts.value.trim()) {
          status.textContent = 'Enter a defect prompt.';
          await new Promise(resolve => setTimeout(resolve, 250));
          continue;
        }
        const t = performance.now();
        const scale = Math.min(1, 1280 / Math.max(video.videoWidth, video.videoHeight));
        canvas.width = Math.round(video.videoWidth * scale);
        canvas.height = Math.round(video.videoHeight * scale);
        context.drawImage(video, 0, 0, canvas.width, canvas.height);
        const result = await google.colab.kernel.invokeFunction(
          'deepsplat.segment_frame', [canvas.toDataURL('image/jpeg', .85), prompts.value], {});
        if (!running) break;
        const url = result.data?.['application/json']?.image;
        if (typeof url !== 'string') throw new Error('Segmentation returned no image. Check the cell output.');
        image.src = url;
        await image.decode();
        status.textContent = Math.round(performance.now() - t) + ' ms per update';
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      }
    } catch (error) {
      status.textContent = error.message;
    } finally {
      stopCamera();
      start.disabled = false;
    }
  };
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
})();
"""))
